# Measuring Spatial Inequality

**DS4DH · Module 09 — Geospatial Analysis**

*Technique:* Coefficient of variation vs standard deviation for comparing dispersion

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/ds4dh-colab-pack/blob/main/notebooks/09c_spatial_inequality.ipynb)

Data: `merged_dataset.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# This notebook reads the CSVs sitting next to it. In Colab, upload them from
# the pack's data/ folder when prompted. The exists() guard means a re-run
# part-way through a session will not ask you to upload all over again.
NEEDED = ['merged_dataset.csv']
missing = [f for f in NEEDED if not os.path.exists(f)]
if missing:
    try:
        from google.colab import files
        print('Upload from the data/ folder of the pack: ' + ', '.join(missing))
        files.upload()
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))

df       = pd.read_csv('merged_dataset.csv')
CITIES = ['Montréal', 'Toronto', 'Edmonton', 'Vancouver']

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

Two cities can have identical average housing burden and completely different
internal structures — one uniform, one sharply divided. The average cannot tell
them apart, and for a housing agency the difference is often what matters.

This notebook measures within-city dispersion, and shows why the obvious measure
(standard deviation) is the wrong one for comparing cities.

In [ ]:
# One row per Census Subdivision.
#   • rows with no csd_code are CMA-level and Canada-level aggregates, not CSDs
#   • each CSD appears 3x (Immigrant / Non-immigrants / Total Immigrant Status)
# Keeping either would silently double- or triple-count places.
csd = df.dropna(subset=['csd_code'])
base = csd[(csd['immigrant_status'] == 'Total Immigrant Status')
           & (csd['cma'].isin(CITIES))].copy()

print(f'{len(df):>4} rows in the raw file')
print(f'{len(csd):>4} after dropping CMA/Canada aggregate rows')
print(f'{len(base):>4} CSDs in the four cities (one row each)')

In [ ]:
d = base.dropna(subset=['Total'])

print(f'{"City":<12}{"n":>5}{"mean":>9}{"sd":>9}{"CV":>9}{"IQR":>9}{"range":>9}')
print('-' * 62)
for city in CITIES:
    s = d[d['cma'] == city]['Total']
    cv = s.std() / s.mean()
    iqr = s.quantile(0.75) - s.quantile(0.25)
    print(f'{city:<12}{len(s):>5}{s.mean():>9.2f}{s.std():>9.2f}'
          f'{cv:>9.3f}{iqr:>9.2f}{s.max() - s.min():>9.2f}')

## Why the coefficient of variation

Standard deviation is in the units of the variable, so it scales with the mean. A
city with twice the average burden will tend to have a larger sd purely
arithmetically, even if it is *relatively* just as uniform.

The **coefficient of variation** (sd ÷ mean) removes that, which makes dispersion
comparable across cities. It is the right default whenever you are comparing
spread between groups with different levels.

In [ ]:
# Demonstrate the scaling artefact directly.
rng = np.random.default_rng(3)
low = rng.normal(15, 3, 200)
high = low * 2          # identical relative spread, doubled level

print(f'{"":<22}{"mean":>9}{"sd":>9}{"CV":>9}')
print('-' * 49)
for name, s in [('low-cost city', low), ('high-cost city', high)]:
    print(f'{name:<22}{s.mean():>9.2f}{s.std():>9.2f}{s.std() / s.mean():>9.3f}')
print()
print('The sd doubled. The CV did not move. Only one of these two numbers')
print('says the cities are equally unequal — which they are, by construction.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

names = [c for c in CITIES if (d['cma'] == c).any()]
axes[0].bar(names, [d[d['cma'] == c]['Total'].std() for c in names], color='#3DA5D9')
axes[0].set_title('Standard deviation (confounded with level)')
axes[0].set_ylabel('sd of Total STIR')

axes[1].bar(names, [d[d['cma'] == c]['Total'].std() / d[d['cma'] == c]['Total'].mean()
                    for c in names], color='#E8663D')
axes[1].set_title('Coefficient of variation (comparable)')
axes[1].set_ylabel('sd / mean')
for ax in axes:
    ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

### 🔧 Your turn 1

Compute the CV for `Renter` and for `tot_income` alongside `Total`.

Which is most unequally distributed within cities? Does the city ranking change
depending on which variable you measure inequality in?

## A distributional view

Dispersion measures compress a whole distribution into one number. Percentile
ratios say something more specific: how far apart are the ends?

In [ ]:
print(f'{"City":<12}{"p10":>8}{"p50":>8}{"p90":>8}{"p90/p10":>10}{"p90-p10":>10}')
print('-' * 56)
for city in CITIES:
    s = d[d['cma'] == city]['Total']
    p10, p50, p90 = s.quantile([0.1, 0.5, 0.9])
    print(f'{city:<12}{p10:>8.1f}{p50:>8.1f}{p90:>8.1f}'
          f'{p90 / p10:>10.2f}{p90 - p10:>10.1f}')
print()
print('The p90/p10 ratio is what a housing agency can act on: it says how much')
print('worse the hardest-pressed municipalities are than the easiest.')

In [ ]:
# Population-weighted dispersion — inequality as people experience it.
dw = d.dropna(subset=['tot_pop'])
print(f'{"City":<12}{"unweighted CV":>16}{"pop-weighted CV":>18}')
print('-' * 46)
for city in CITIES:
    g = dw[dw['cma'] == city]
    m = np.average(g['Total'], weights=g['tot_pop'])
    var = np.average((g['Total'] - m) ** 2, weights=g['tot_pop'])
    print(f'{city:<12}{g["Total"].std() / g["Total"].mean():>16.3f}'
          f'{np.sqrt(var) / m:>18.3f}')
print()
print('Where the weighted CV is much smaller, the extreme municipalities are')
print('small ones — the inequality is real but affects relatively few people.')

### 🔧 Your turn 2

Compare the unweighted and population-weighted CV columns.

Pick the city where they differ most. Write the sentence you would use to
describe its internal inequality without overstating how many households it
affects.

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** Income is typically the most unequally distributed of the three,
and renter STIR more dispersed than total STIR. The city ranking does change
depending on the variable — which is why "the most unequal city" is not a
well-formed claim until you say unequal *in what*. Naming the variable is not
pedantry; it is the difference between a measurable statement and a slogan.

**Your turn 2.** For the city with the biggest gap between the two CVs, something
like:

> Housing burden varies widely across this metropolitan area's municipalities
> (CV = 0.24), but the most extreme values occur in small subdivisions; weighting
> by population reduces the measured dispersion to 0.15. The inequality is real
> at the municipal level while affecting a modest share of households.

That sentence is harder to write than "this is the most unequal city" and it is
the one that survives scrutiny.

</details>

## Where this stops

Module 09's conclusion: you can do real spatial analysis without a map, and the
boundaries you were handed are a choice you inherited rather than a fact.

Next: Module 10 is about showing all of this to someone who will not read your code.